In [0]:
# ============================================================
# NOTEBOOK: nb_00_Setup
# PURPOSE:  Creates all schemas, audit tables, configuration
#           tables, and mapping tables required by the KTU
#           assessment pipeline.
#
#           This notebook is idempotent. It is safe to re-run.
#           It does not ingest or transform source data.
#
# CATALOG:  ktu_assessment_dev
# COMPUTE:  Serverless
# INPUT:    None
# OUTPUT:   Audit, configuration, and mapping tables
# ============================================================

# ------------------------------------------------------------
# IMPORTS
# uuid     = generates the run_id used across all notebooks
# datetime = captures timestamps for audit rows
# ------------------------------------------------------------
import uuid
from datetime import datetime

# ------------------------------------------------------------
# CONFIGURATION
# Every name used by this notebook and by downstream
# notebooks is defined here. No paths or table names are
# hard-coded anywhere else in the pipeline.
# ------------------------------------------------------------

CATALOG       = "ktu_assessment_dev"
AUDIT_SCHEMA  = "audit"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"

VOLUME_PATH   = "/Volumes/ktu_assessment_dev/bronze/raw_landing"
SOURCE_ROOT   = f"{VOLUME_PATH}/Training Data"

# Debug flag. When 1, prints progress banners. When 0,
# produces only the final summary. The audit tables are
# written regardless of this flag.
DEBUG = 1

# ------------------------------------------------------------
# RUN HEADER
# The run_id is generated once here and reused by every
# downstream notebook in this execution.
# ------------------------------------------------------------
run_id     = str(uuid.uuid4())
notebook   = "nb_00_Setup"
start_time = datetime.now()

if DEBUG:
    print("=" * 50)
    print("NB_00_SETUP STARTED")
    print("=" * 50)
    print(f"Run ID     : {run_id}")
    print(f"Catalog    : {CATALOG}")
    print(f"Volume     : {VOLUME_PATH}")
    print(f"Start Time : {start_time}")

NB_00_SETUP STARTED
Run ID     : dd79e1fa-3966-49cd-86e4-701c4689ca66
Catalog    : ktu_assessment_dev
Volume     : /Volumes/ktu_assessment_dev/bronze/raw_landing
Start Time : 2026-09-12 12:28:14.327154


In [0]:
# ============================================================
# AUDIT TABLES
# ============================================================
#
# WHAT THIS CELL DOES:
# Creates the four tables that record what the pipeline did,
# what files it processed, what schema it expected, and what
# data-quality checks were run.
#
# WHY THESE FOUR TABLES:
#   pipeline_run_log      - answers "what ran and did it succeed"
#   file_manifest         - answers "what source files were seen"
#                           and enforces idempotency via fingerprint
#   schema_registry       - answers "what columns did we expect"
#                           and drives schema validation
#   data_quality_results  - answers "what checks passed or failed"
#
# WHY CREATE TABLE IF NOT EXISTS:
# Setup must be rerunnable without error. If a table already
# exists, we leave it alone. Resetting content is a separate
# concern handled by explicit DELETE statements where needed.
#
# WHY DELTA:
# Delta provides ACID guarantees, time travel, and schema
# enforcement. Required for reliable audit and idempotency.
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log (
        run_id           STRING    COMMENT 'UUID generated per notebook execution',
        notebook_name    STRING    COMMENT 'Notebook that produced this row',
        layer            STRING    COMMENT 'bronze, silver, gold, or audit',
        target_object    STRING    COMMENT 'Table or volume path processed',
        start_time       TIMESTAMP COMMENT 'Notebook start timestamp',
        end_time         TIMESTAMP COMMENT 'Notebook end timestamp',
        status           STRING    COMMENT 'RUNNING, SUCCESS, or FAILED',
        rows_in          BIGINT    COMMENT 'Rows read or received',
        rows_out         BIGINT    COMMENT 'Rows written or produced',
        rows_rejected    BIGINT    COMMENT 'Rows quarantined or excluded',
        message          STRING    COMMENT 'Free-text status or error message',
        duration_seconds INT       COMMENT 'Elapsed seconds end minus start'
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.file_manifest (
        file_name        STRING    COMMENT 'Basename of the source file',
        file_path        STRING    COMMENT 'Full volume path',
        file_size_bytes  BIGINT    COMMENT 'File size at ingest',
        fingerprint      STRING    COMMENT 'SHA-256 of file content',
        first_seen_at    TIMESTAMP COMMENT 'First time this file was ingested',
        last_ingested_at TIMESTAMP COMMENT 'Most recent ingestion of this file',
        ingest_count     INT       COMMENT 'Number of times this file was ingested',
        source_name      STRING    COMMENT 'Logical source name from ingestion_config'
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.schema_registry (
        source_name      STRING    COMMENT 'Logical source name',
        sheet_name       STRING    COMMENT 'Sheet within the source, or (csv)',
        column_name      STRING    COMMENT 'Exact column name as expected in source',
        data_type        STRING    COMMENT 'Expected Spark data type',
        is_required      BOOLEAN   COMMENT 'True if the column must be present',
        is_active        BOOLEAN   COMMENT 'False to disable validation for this column',
        notes            STRING    COMMENT 'Optional explanation of unusual structure'
    )
    USING DELTA
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.data_quality_results (
        run_id           STRING    COMMENT 'Pipeline run ID',
        check_name       STRING    COMMENT 'Short identifier of the check',
        check_category   STRING    COMMENT 'schema, completeness, uniqueness, validity, consistency, reconciliation',
        target_object    STRING    COMMENT 'Table or column the check ran against',
        expected         STRING    COMMENT 'Expected result, human-readable',
        actual           STRING    COMMENT 'Actual result, human-readable',
        status           STRING    COMMENT 'PASS, FAIL, or WARN',
        message          STRING    COMMENT 'Additional detail if status is FAIL or WARN',
        check_ts         TIMESTAMP COMMENT 'When the check ran'
    )
    USING DELTA
""")

if DEBUG:
    print("=" * 50)
    print("AUDIT TABLES CREATED")
    print("=" * 50)
    for t in ["pipeline_run_log", "file_manifest", "schema_registry", "data_quality_results"]:
        print(f"  {CATALOG}.{AUDIT_SCHEMA}.{t}")

AUDIT TABLES CREATED
  ktu_assessment_dev.audit.pipeline_run_log
  ktu_assessment_dev.audit.file_manifest
  ktu_assessment_dev.audit.schema_registry
  ktu_assessment_dev.audit.data_quality_results


In [0]:
# ============================================================
# INGESTION CONFIGURATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Creates the metadata table that drives the ingestion
# notebook. One row per logical source. The ingestion
# notebook reads this table to decide what to process.
#
# WHY THIS EXISTS:
# Without it, ingestion logic would need to hard-code six
# different source structures in Python. With it, the
# ingestion framework is generic and the source-specific
# differences (header row, sheet name, format) are data
# rows rather than code branches.
#
# WHY METADATA IN A TABLE AND NOT A PYTHON DICT:
# A table can be updated without redeploying code. It is
# queryable by a reviewer. It survives notebook restarts.
# It documents the pipeline's source contract in one place.
#
# COLUMNS:
#   source_name       - logical identifier used throughout the pipeline
#   source_file       - basename as it appears in the volume
#   source_subfolder  - subfolder under SOURCE_ROOT, or NULL
#   source_format     - xlsx or csv
#   sheet_name        - sheet to read, or (csv) for CSVs
#   header_row        - zero-based row index containing column names
#   target_table      - Bronze table name to write into
#   target_layer      - always bronze for ingestion outputs
#   processing_order  - execution order for the ingestion loop
#   active_flag       - include in the current run
#   notes             - structural quirks worth documenting
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.ingestion_config (
        source_name       STRING    COMMENT 'Logical source identifier',
        source_file       STRING    COMMENT 'Basename of file in the volume',
        source_subfolder  STRING    COMMENT 'Relative folder under SOURCE_ROOT, NULL for root',
        source_format     STRING    COMMENT 'xlsx or csv',
        sheet_name        STRING    COMMENT 'Sheet name, or (csv) for delimited text',
        header_row        INT       COMMENT 'Zero-based header row index',
        target_table      STRING    COMMENT 'Target Bronze table name',
        target_layer      STRING    COMMENT 'bronze, silver, or gold',
        processing_order  INT       COMMENT 'Execution order within the layer',
        active_flag       BOOLEAN   COMMENT 'Include this source in the current run',
        notes             STRING    COMMENT 'Structural quirks or special handling'
    )
    USING DELTA
""")

# ------------------------------------------------------------
# SEED INGESTION CONFIG
# Delete any existing rows to make this cell idempotent,
# then insert the current source definitions.
# ------------------------------------------------------------
spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.ingestion_config")

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.ingestion_config
    VALUES
    ('capturing_tool',        'Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx',
                              NULL,
                              'xlsx', 'Capturer1', 0,
                              'capturing_tool_capturer1', 'bronze', 10, true,
                              'Three sheets unioned into one logical source; 37 columns per sheet'),

    ('capturing_tool',        'Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx',
                              NULL,
                              'xlsx', 'Capturer2', 0,
                              'capturing_tool_capturer2', 'bronze', 11, true,
                              'Second capturer sheet, same schema as Capturer1'),

    ('capturing_tool',        'Capturing Tool_V1c V2-27 January 2026_SCRUBBED.xlsx',
                              NULL,
                              'xlsx', 'Capturer3', 0,
                              'capturing_tool_capturer3', 'bronze', 12, true,
                              'Third capturer sheet, same schema as Capturer1'),

    ('chw_west_coast',        'CHW Training Attendance-West Coast-Oct 2025_SCRUBBED.xlsx',
                              'Community Health Worker Completions',
                              'xlsx', 'Training Attendance', 1,
                              'chw_west_coast_attendance', 'bronze', 20, true,
                              'header_row=1 because row 0 is the printed form title; 49 columns; 15 module columns'),

    ('chw_kess',              'CHW Training Attendance_KESS_December 2025_SCRUBBED.xlsx',
                              'Community Health Worker Completions',
                              'xlsx', 'Training Attendance', 1,
                              'chw_kess_attendance', 'bronze', 21, true,
                              'Same schema as CHW West Coast; header_row=1'),

    ('chw_witzenberg',        'WITZENBERG - July-Dec 2025_SCRUBBED.xlsx',
                              'Community Health Worker Completions',
                              'xlsx', 'Training Attendance', 1,
                              'chw_witzenberg_attendance', 'bronze', 22, true,
                              'Same schema as CHW West Coast; header_row=1'),

    ('online_export',         'Online Data Export 17 Dec_SCRUBBED.csv',
                              NULL,
                              'csv', '(csv)', 0,
                              'online_export', 'bronze', 30, true,
                              'UTF-8, comma-delimited, 27 columns, system-generated'),

    ('lookup_courses',        'Course and Facility Look Ups.xlsx',
                              NULL,
                              'xlsx', 'LU_Courses', 0,
                              'lookup_courses', 'bronze', 40, true,
                              'Course reference data; 340 rows, 7 columns'),

    ('lookup_facility',       'Course and Facility Look Ups.xlsx',
                              NULL,
                              'xlsx', 'LU_Facility', 0,
                              'lookup_facility', 'bronze', 41, true,
                              'Authoritative facility reference; 809 rows, 19 columns')
""")

if DEBUG:
    count = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{AUDIT_SCHEMA}.ingestion_config").collect()[0]["n"]
    print("=" * 50)
    print("INGESTION CONFIG SEEDED")
    print("=" * 50)
    print(f"Rows in config : {count}")
    print()
    spark.sql(f"""
        SELECT source_name, sheet_name, target_table, processing_order
        FROM {CATALOG}.{AUDIT_SCHEMA}.ingestion_config
        ORDER BY processing_order
    """).show(truncate=False)

INGESTION CONFIG SEEDED
Rows in config : 9

+---------------+-------------------+-------------------------+----------------+
|source_name    |sheet_name         |target_table             |processing_order|
+---------------+-------------------+-------------------------+----------------+
|capturing_tool |Capturer1          |capturing_tool_capturer1 |10              |
|capturing_tool |Capturer2          |capturing_tool_capturer2 |11              |
|capturing_tool |Capturer3          |capturing_tool_capturer3 |12              |
|chw_west_coast |Training Attendance|chw_west_coast_attendance|20              |
|chw_kess       |Training Attendance|chw_kess_attendance      |21              |
|chw_witzenberg |Training Attendance|chw_witzenberg_attendance|22              |
|online_export  |(csv)              |online_export            |30              |
|lookup_courses |LU_Courses         |lookup_courses           |40              |
|lookup_facility|LU_Facility        |lookup_facility          |41

In [0]:
# ============================================================
# PROFESSIONAL CATEGORY MAPPING
# ============================================================
#
# WHAT THIS CELL DOES:
# Loads the mapping that reduces the 54 typo variants found
# in the Capturing Tool 'Professional Category' column into
# 4 canonical values: Doctor, Nurse, Pharmacist, Other.
#
# WHY THIS IS A MAPPING TABLE AND NOT A CASE STATEMENT:
# The assessment's data contains deliberately planted typos.
# The correct engineering response is a controlled mapping
# table that is auditable and extendable, not a fragile
# inline string comparison.
#
# WHY THE VARIANTS ARE LISTED EXPLICITLY:
# String similarity would wrongly merge things like
# 'Nurse' and 'Nursery'. Exact-match mapping on a controlled
# list is deterministic and reviewable. Any value not in this
# list will be surfaced as unmapped in Silver, not silently
# coerced.
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.map_professional_category (
        source_value       STRING COMMENT 'Raw value as read from source',
        canonical_value    STRING COMMENT 'Doctor, Nurse, Pharmacist, or Other',
        mapping_notes      STRING COMMENT 'Why this mapping exists'
    )
    USING DELTA
""")

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.map_professional_category")

# ------------------------------------------------------------
# Build the mapping programmatically for the typo families.
# This keeps the seed code compact while producing an
# explicit, reviewable mapping table.
# ------------------------------------------------------------
doctor_variants = [
    "Doctor", "doctor", "DOCTOR",
    "Dcotor", "Dctor", "Docctor", "Docor", "Docotr", "Doctoor",
    "Doctr", "Doctro", "Docttor", "Dooctor", "Dotcor", "Dotor",
    " Doctor ",
]
nurse_variants = [
    "Nurse", "nurse", "NURSE",
    "Nrse", "Nruse", "Nure", "Nures", "Nurrse", "Nursse",
    "Nuse", "Nusre", "Nuurse",
    " Nurse ",
]
pharmacist_variants = [
    "Pharmacist", "pharmacist", "PHARMACIST",
    "Pahrmacist", "Pharmaacist", "Pharmaccist", "Pharmacit",
    "Pharmaicst", "Pharmcaist", "Pharrmacist", "Phharmacist",
    " Pharmacist ",
]
other_variants = [
    "Other", "other", "OTHER",
    "Oher", "Ohter", "Otehr", "Oter", "Otheer", "Othher",
    "Othr", "Othre", "Otther",
    " Other ",
]

mapping_rows = []
for v in doctor_variants:
    mapping_rows.append((v, "Doctor", "Typo or case variant of Doctor"))
for v in nurse_variants:
    mapping_rows.append((v, "Nurse", "Typo or case variant of Nurse"))
for v in pharmacist_variants:
    mapping_rows.append((v, "Pharmacist", "Typo or case variant of Pharmacist"))
for v in other_variants:
    mapping_rows.append((v, "Other", "Typo or case variant of Other"))

mapping_df = spark.createDataFrame(
    mapping_rows,
    schema="source_value STRING, canonical_value STRING, mapping_notes STRING"
)

mapping_df.write.mode("append").saveAsTable(
    f"{CATALOG}.{AUDIT_SCHEMA}.map_professional_category"
)

if DEBUG:
    print("=" * 50)
    print("PROFESSIONAL CATEGORY MAPPING SEEDED")
    print("=" * 50)
    spark.sql(f"""
        SELECT canonical_value, COUNT(*) AS variant_count
        FROM {CATALOG}.{AUDIT_SCHEMA}.map_professional_category
        GROUP BY canonical_value
        ORDER BY canonical_value
    """).show(truncate=False)

PROFESSIONAL CATEGORY MAPPING SEEDED
+---------------+-------------+
|canonical_value|variant_count|
+---------------+-------------+
|Doctor         |16           |
|Nurse          |13           |
|Other          |13           |
|Pharmacist     |12           |
+---------------+-------------+



In [0]:
# ============================================================
# DISTRICT MAPPING
# ============================================================
#
# WHAT THIS CELL DOES:
# Loads the mapping that reduces the 117 typo variants found
# in the Capturing Tool 'District' column into the 7
# canonical districts defined in the CHW embedded lookup:
#   Cape Winelands, Central Karoo, Garden Route, Metro,
#   Overberg, West Coast, Other
#
# WHY THESE 7 CANONICAL VALUES:
# The embedded lu_District sheet in every CHW workbook
# defines exactly these 7 values. They are the source of
# truth for district naming.
#
# HOW THE CSV DISTRICTS ARE HANDLED:
# The CSV uses a different format: 'Cape Winelands District
# Municipality'. These are handled separately in Silver by
# stripping the suffix and matching case-insensitively. They
# are not seeded here because they are not typos - they are
# a different naming convention that follows a predictable
# rule.
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.map_district (
        source_value       STRING COMMENT 'Raw value as read from source',
        canonical_value    STRING COMMENT 'Canonical district name',
        mapping_notes      STRING COMMENT 'Why this mapping exists'
    )
    USING DELTA
""")

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.map_district")

# ------------------------------------------------------------
# Build the mapping programmatically for each district family.
# Every typo variant observed in the profiling is included.
# The canonical value is the value used in the CHW lookup.
# ------------------------------------------------------------
district_families = {
    "Cape Winelands": [
        "Cape Winelands", "CAPE WINELANDS", " Cape Winelands ",
        "Caape Winelands", "Cae Winelands", "Caep Winelands",
        "Cap eWinelands", "Cape  Winelands", "Cape Wielands",
        "Cape Wienlands", "Cape Wiinelands", "Cape Wineands",
        "Cape Wineelands", "Cape Winelads", "Cape Winelandds",
        "Cape Winelannds", "Cape Winelans", "Cape Winellands",
        "Cape Winelnads", "Cape Winlands", "Cape Winleands",
        "Cape Winnelands", "Cape Wnelands", "Cape Wnielands",
        "CapeW inelands", "Capee Winelands", "Cpae Winelands",
        "Cpe Winelands",
    ],
    "Central Karoo": [
        "Central Karoo", "CENTRAL KAROO", " Central Karoo ",
        "Cenrtal Karoo", "Centra lKaroo", "Central  Karoo",
        "Central Kaaroo", "Central Karo", "Central Karroo",
        "Centrral Karoo", "Centtral Karoo", "Cnetral Karoo",
    ],
    "Garden Route": [
        "Garden Route", "GARDEN ROUTE", " Garden Route ",
        "Gaarden Route", "Gaden Route", "Gadren Route",
        "Gardden Route", "Garde Route", "Garde nRoute",
        "Gardeen Route", "Garden  Route", "Garden RRoute",
        "Garden Rooute", "Garden Rote", "Garden Rotue",
        "Garden Roue", "Garden Rouet",
    ],
    "Metro": [
        "Metro", "METRO", " Metro ",
    ],
    "Overberg": [
        "Overberg", "OVERBERG", " Overberg ",
    ],
    "West Coast": [
        "West Coast", "WEST COAST", " West Coast ",
    ],
    "Other": [
        "Other", "OTHER", " Other ",
    ],
}

district_rows = []
for canonical, variants in district_families.items():
    for v in variants:
        district_rows.append((v, canonical, f"Variant of {canonical}"))

district_df = spark.createDataFrame(
    district_rows,
    schema="source_value STRING, canonical_value STRING, mapping_notes STRING"
)

district_df.write.mode("append").saveAsTable(
    f"{CATALOG}.{AUDIT_SCHEMA}.map_district"
)

if DEBUG:
    print("=" * 50)
    print("DISTRICT MAPPING SEEDED")
    print("=" * 50)
    spark.sql(f"""
        SELECT canonical_value, COUNT(*) AS variant_count
        FROM {CATALOG}.{AUDIT_SCHEMA}.map_district
        GROUP BY canonical_value
        ORDER BY canonical_value
    """).show(truncate=False)

DISTRICT MAPPING SEEDED
+---------------+-------------+
|canonical_value|variant_count|
+---------------+-------------+
|Cape Winelands |28           |
|Central Karoo  |12           |
|Garden Route   |17           |
|Metro          |3            |
|Other          |3            |
|Overberg       |3            |
|West Coast     |3            |
+---------------+-------------+



In [0]:
# ============================================================
# EMPLOYER GROUP MAPPING
# ============================================================
#
# WHAT THIS CELL DOES:
# Maps employer group values across the three sources to a
# common canonical set.
#
# WHY THESE CROSS-SOURCE MAPPINGS ARE NEEDED:
# Each source uses different terminology for the same
# employer categories:
#
#   Capturing Tool    CHW       CSV                    Canonical
#   ---------------   -------   -------------------    ---------------
#   NDOH              -         Department of Health   National DOH
#   CoCT              -         Municipality (eg.City) Municipality
#   NGO/NPO           NGO/NPO   NGO/NPO/DSP            NGO/NPO
#   Private           -         Private Sector         Private
#   WCGHW             -         -                      WCGHW
#   Correctional...   -         -                      Correctional Services
#   HEI               -         -                      HEI
#   Other             -         Other                  Other
#   -                 -         Other - Security Co.  Other
#
# CANONICAL SET:
# Chosen to preserve the distinctions that the source systems
# actually make, without inventing categories. Nine values.
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.map_employer_group (
        source_value       STRING COMMENT 'Raw value from source',
        source_system      STRING COMMENT 'capturing_tool, chw, or online_export',
        canonical_value    STRING COMMENT 'Common canonical employer group',
        mapping_notes      STRING COMMENT 'Why this mapping exists'
    )
    USING DELTA
""")

spark.sql(f"DELETE FROM {CATALOG}.{AUDIT_SCHEMA}.map_employer_group")

employer_rows = [
    ("NDOH",                          "capturing_tool", "National DOH",             "Capturing Tool abbreviation"),
    ("Department of Health",          "online_export",  "National DOH",             "CSV full name"),
    ("CoCT",                          "capturing_tool", "Municipality",             "City of Cape Town"),
    ("Municipality (eg.City)",        "online_export",  "Municipality",             "CSV generic municipality"),
    ("NGO/NPO",                       "capturing_tool", "NGO/NPO",                  "Direct"),
    ("NGO/NPO",                       "chw",            "NGO/NPO",                  "Direct"),
    ("NGO/NPO/DSP",                   "online_export",  "NGO/NPO",                  "CSV combined value"),
    ("Private",                       "capturing_tool", "Private",                  "Direct"),
    ("Private Sector",                "online_export",  "Private",                  "CSV full name"),
    ("WCGHW",                         "capturing_tool", "WCGHW",                    "Western Cape Government Health and Wellness"),
    ("Correctional Services (DCS)",   "capturing_tool", "Correctional Services",    "Direct"),
    ("HEI",                           "capturing_tool", "HEI",                      "Higher Education Institution"),
    ("Other",                         "capturing_tool", "Other",                    "Direct"),
    ("Other",                         "online_export",  "Other",                    "Direct"),
    ("Other - Security Company",      "online_export",  "Other",                    "Mapped to Other per canonical set"),
]

employer_df = spark.createDataFrame(
    employer_rows,
    schema="source_value STRING, source_system STRING, canonical_value STRING, mapping_notes STRING"
)

employer_df.write.mode("append").saveAsTable(
    f"{CATALOG}.{AUDIT_SCHEMA}.map_employer_group"
)

if DEBUG:
    print("=" * 50)
    print("EMPLOYER GROUP MAPPING SEEDED")
    print("=" * 50)
    spark.sql(f"""
        SELECT canonical_value, COUNT(*) AS source_variants
        FROM {CATALOG}.{AUDIT_SCHEMA}.map_employer_group
        GROUP BY canonical_value
        ORDER BY canonical_value
    """).show(truncate=False)

EMPLOYER GROUP MAPPING SEEDED
+---------------------+---------------+
|canonical_value      |source_variants|
+---------------------+---------------+
|Correctional Services|1              |
|HEI                  |1              |
|Municipality         |2              |
|NGO/NPO              |3              |
|National DOH         |2              |
|Other                |3              |
|Private              |2              |
|WCGHW                |1              |
+---------------------+---------------+



In [0]:
# ============================================================
# FACILITY MAPPING TABLE
# ============================================================
#
# WHAT THIS CELL DOES:
# Creates an empty facility mapping table. It is populated in
# the Bronze layer from the standalone LU_Facility lookup,
# not seeded here.
#
# WHY EMPTY AND NOT SEEDED:
# The facility reference data lives in 'Course and Facility
# Look Ups.xlsx' and totals 809 rows. Reproducing that seed
# here would duplicate source-of-truth data and risk drift.
# The correct pattern is to load the lookup in Bronze, then
# populate this mapping table from Bronze in Silver.
#
# WHY THIS TABLE EXISTS AT ALL:
# Facility names appear differently across sources:
#   Capturing Tool: ' FALSE BAY HOSPITAL ' (uppercase, spaced)
#   CHW:            'BELLA VISTA CLINIC'  (uppercase)
#   CSV:            'False Bay Hospital'  (title case)
#   Lookup:         'False Bay Hospital'  (title case)
#
# The mapping table stores, per source value, which canonical
# facility it resolves to. It also records which lookup the
# match came from (standalone, embedded, or none).
#
# UNMATCHED FACILITIES:
# Any source value that does not match any lookup will be
# recorded here with facility_code NULL and match_status
# UNMATCHED. The assessment requires unmapped items to be
# surfaced, not dropped.
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.map_facility (
        source_value        STRING COMMENT 'Raw facility value from any source',
        canonical_facility  STRING COMMENT 'Matched FacilityName from lookup',
        facility_code       STRING COMMENT 'FacilityCode from lookup, NULL if unmatched',
        health_district     STRING COMMENT 'HealthDistrict from lookup',
        health_subdistrict  STRING COMMENT 'HealthSubdistrict from lookup',
        match_source        STRING COMMENT 'standalone_lookup, embedded_lookup, or UNMATCHED',
        match_notes         STRING COMMENT 'How the match was made'
    )
    USING DELTA
""")

if DEBUG:
    print("=" * 50)
    print("FACILITY MAPPING TABLE CREATED (empty)")
    print("=" * 50)
    print(f"  {CATALOG}.{AUDIT_SCHEMA}.map_facility")
    print("Populated in Bronze/Silver from the facility lookup.")

FACILITY MAPPING TABLE CREATED (empty)
  ktu_assessment_dev.audit.map_facility
Populated in Bronze/Silver from the facility lookup.


In [0]:
# ============================================================
# PROFESSION MAPPING TABLE
# ============================================================
#
# WHAT THIS CELL DOES:
# Creates an empty profession mapping table. It is populated
# in Silver from the distinct profession values across the
# three sources.
#
# WHY NOT SEEDED WITH VALUES:
# The three sources use three different profession
# taxonomies:
#   Capturing Tool:  56 job titles with abbreviations
#   CHW:             2 values
#   CSV:             22 values
#
# The correct canonical profession set depends on how we
# decide to unify these. That decision is made in Silver
# after reviewing the full distinct list side by side.
# Seeding now would be premature.
#
# WHAT GOES HERE EVENTUALLY:
#   source_value         - the raw value from any source
#   source_system        - which source it appeared in
#   canonical_profession - the standardised profession
#   profession_group     - clinical, nursing, allied, non-clinical
# ============================================================

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{AUDIT_SCHEMA}.map_profession (
        source_value         STRING COMMENT 'Raw profession value from source',
        source_system        STRING COMMENT 'capturing_tool, chw, or online_export',
        canonical_profession STRING COMMENT 'Standardised profession name',
        profession_group     STRING COMMENT 'Clinical, Nursing, Allied, Non-clinical, Other',
        mapping_notes        STRING COMMENT 'Why this mapping exists'
    )
    USING DELTA
""")

if DEBUG:
    print("=" * 50)
    print("PROFESSION MAPPING TABLE CREATED (empty)")
    print("=" * 50)
    print(f"  {CATALOG}.{AUDIT_SCHEMA}.map_profession")
    print("Populated in Silver after source profession review.")

PROFESSION MAPPING TABLE CREATED (empty)
  ktu_assessment_dev.audit.map_profession
Populated in Silver after source profession review.


In [0]:
# ============================================================
# SETUP VERIFICATION
# ============================================================
#
# WHAT THIS CELL DOES:
# Confirms every table created by this notebook exists and
# reports its row count. If any table is missing, the
# downstream notebooks will fail immediately, so this check
# must pass before the pipeline is considered ready.
# ============================================================

tables_to_verify = [
    (AUDIT_SCHEMA,  "pipeline_run_log"),
    (AUDIT_SCHEMA,  "file_manifest"),
    (AUDIT_SCHEMA,  "schema_registry"),
    (AUDIT_SCHEMA,  "data_quality_results"),
    (AUDIT_SCHEMA,  "ingestion_config"),
    (AUDIT_SCHEMA,  "map_professional_category"),
    (AUDIT_SCHEMA,  "map_district"),
    (AUDIT_SCHEMA,  "map_employer_group"),
    (AUDIT_SCHEMA,  "map_facility"),
    (AUDIT_SCHEMA,  "map_profession"),
]

verification_rows = []
for schema, table in tables_to_verify:
    full_name = f"{CATALOG}.{schema}.{table}"
    try:
        count = spark.sql(f"SELECT COUNT(*) AS n FROM {full_name}").collect()[0]["n"]
        verification_rows.append((full_name, count, "OK"))
    except Exception as exc:
        verification_rows.append((full_name, None, f"ERROR: {exc}"))

if DEBUG:
    print("=" * 70)
    print("SETUP VERIFICATION")
    print("=" * 70)
    print(f"{'TABLE':<60} {'ROWS':>8}  STATUS")
    print("-" * 70)
    for name, count, status in verification_rows:
        count_str = str(count) if count is not None else "-"
        print(f"{name:<60} {count_str:>8}  {status}")
    print("=" * 70)

# ------------------------------------------------------------
# CLEAN UP THE SMOKE TEST TABLE FROM THE ENVIRONMENT CHECK
# ------------------------------------------------------------
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{AUDIT_SCHEMA}.smoke_test")

# ------------------------------------------------------------
# WRITE A PIPELINE RUN LOG ENTRY
# ------------------------------------------------------------
end_time = datetime.now()
duration = int((end_time - start_time).total_seconds())

spark.sql(f"""
    INSERT INTO {CATALOG}.{AUDIT_SCHEMA}.pipeline_run_log
    VALUES (
        '{run_id}',
        '{notebook}',
        'audit',
        'setup',
        '{start_time.strftime("%Y-%m-%d %H:%M:%S")}',
        '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
        'SUCCESS',
        {len(tables_to_verify)},
        {len([r for r in verification_rows if r[2] == "OK"])},
        0,
        'Setup notebook completed. All metadata tables created.',
        {duration}
    )
""")

if DEBUG:
    print()
    print("=" * 50)
    print("NB_00_SETUP COMPLETED")
    print("=" * 50)
    print(f"Run ID           : {run_id}")
    print(f"Tables verified  : {len(tables_to_verify)}")
    print(f"Tables OK        : {len([r for r in verification_rows if r[2] == 'OK'])}")
    print(f"Duration         : {duration}s")
    print("=" * 50)

SETUP VERIFICATION
TABLE                                                            ROWS  STATUS
----------------------------------------------------------------------
ktu_assessment_dev.audit.pipeline_run_log                           0  OK
ktu_assessment_dev.audit.file_manifest                              0  OK
ktu_assessment_dev.audit.schema_registry                            0  OK
ktu_assessment_dev.audit.data_quality_results                       0  OK
ktu_assessment_dev.audit.ingestion_config                           9  OK
ktu_assessment_dev.audit.map_professional_category                 54  OK
ktu_assessment_dev.audit.map_district                              69  OK
ktu_assessment_dev.audit.map_employer_group                        15  OK
ktu_assessment_dev.audit.map_facility                               0  OK
ktu_assessment_dev.audit.map_profession                             0  OK

NB_00_SETUP COMPLETED
Run ID           : dd79e1fa-3966-49cd-86e4-701c4689ca66
Tables verifi